# Use of Force Data Cleaning

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/use_of_force.csv")

In [2]:
df.dtypes

Incident Number              int64
Subject_Incident Number    float64
Sex                            str
Race                           str
Subject Role                   str
Subject Age                float64
Block Address                  str
City                           str
Use of Force Number          int64
Officer_Incident Number      int64
Officer Role                   str
Use of Force Level             str
Use of Force Type              str
OccurredDate                   str
Subject Arrested               str
Use of Force Reason            str
Subject Altered State          str
NonBerkeley Address           bool
dtype: object

## Standardize Column Names & ID Types

In [3]:
# rename columns
rename_map = {"Incident Number": "incident_number", "Subject_Incident Number": "subject_incident_number", 
              "Sex": "sex", "Race": "race", "Subject Role": "subject_role", "Subject Age": "subject_age",
              "Block Address": "block_address", "City": "city", "Use of Force Number": "use_of_force_number",
              "Officer_Incident Number": "officer_incident_number", "Officer Role": "officer_role", 
              "Use of Force Level": "use_of_force_level", "Use of Force Type": "use_of_force_type",
              "OccurredDate": "occurred_date", "Subject Arrested": "subject_arrested", 
              "Use of Force Reason": "use_of_force_reason", "Subject Altered State": "subject_altered_state",
              "NonBerkeley Address": "non_berkeley_address"}
df = df.rename(columns=rename_map)

# convert ID columns to nullable integers 
id_cols = ["incident_number", "subject_incident_number", "use_of_force_number", "officer_incident_number"]
for col in id_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

df.head()

,incident_number,subject_incident_number,sex,race,subject_role,subject_age,block_address,city,use_of_force_number,officer_incident_number,officer_role,use_of_force_level,use_of_force_type,occurred_date,subject_arrested,use_of_force_reason,subject_altered_state,non_berkeley_address
0,4475,3073,Male,Black,Arrested,45.0,700 Block of Folger,Berkeley,8647,10524,Cover Officer,Level 2,Level 2 Handgun DNP,2026-11-22,Yes,Reported as Armed,None detected,False
1,4475,3073,Male,Black,Arrested,45.0,700 Block of Folger,Berkeley,8646,10523,Primary Officer,Level 2,Level 2 Handgun Pointed,2026-11-22,Yes,Reported as Armed,None detected,False
2,4475,3072,Male,Black,Arrested,46.0,700 Block of Folger,Berkeley,8647,10524,Cover Officer,Level 2,Level 2 Handgun DNP,2026-11-22,Yes,Reported as Armed,None detected,False
3,4475,3072,Male,Black,Arrested,46.0,700 Block of Folger,Berkeley,8646,10523,Primary Officer,Level 2,Level 2 Handgun Pointed,2026-11-22,Yes,Reported as Armed,None detected,False
4,4477,3074,Female,White,Suspect,18.0,2600 Block of Piedmont,Berkeley,8648,10526,Primary Officer,Level 1,Level 1 Grab,2026-11-21,No,Fleeing,Alcohol,False


## Filter to Berkeley & Timeframe (2021-03-01 to 2026-04-30)

In [4]:
# filter city to Berkeley
df["city"] = df["city"].astype(str).str.strip().str.title()
df = df[df["city"] == "Berkeley"].copy()

# filter timeframe
dt_temp = pd.to_datetime(df["occurred_date"], utc=True, errors="coerce")
start_ts = pd.Timestamp("2021-03-01", tz="UTC")
end_ts = pd.Timestamp("2026-04-30 23:59:59", tz="UTC")

df = df[(dt_temp >= start_ts) & (dt_temp <= end_ts)].copy()
dt_temp = dt_temp[(dt_temp >= start_ts) & (dt_temp <= end_ts)]

# format occurred_date to match team spec & extract temporal helper columns
df["occurred_date"] = dt_temp.dt.strftime("%Y-%m-%d %H:%M:%S+00")

# --- verification check ---
print(f"Filtered row count: {len(df):,}")
print(f"Unique City: {df["city"].unique().tolist()}")
print(f"Date range: {df['occurred_date'].min()} to {df['occurred_date'].max()}")

Filtered row count: 7,530
Unique City: ['Berkeley']
Date range: 2021-03-01 00:00:00+00 to 2026-04-30 00:00:00+00


## Categorical & Text Cleanup

In [5]:
na_tokens = ["Unknown", "Not Appl", "Not Applicable", "nan", "None", ""]

# clean Use of Force Level & create numeric version (1-4)
df["use_of_force_level"] = (df["use_of_force_level"].astype(str).str.strip().replace(na_tokens, pd.NA))

level_map = {"Level 1": 1, "Level 2": 2, "Level 3": 3, "Level 4": 4}
df["force_level_num"] = df["use_of_force_level"].map(level_map).astype("Int64")

# clean Use of Force Type typos/spacing & extract force technique excluding the level
df["use_of_force_type"] = (df["use_of_force_type"].astype(str).str.strip()
    .str.replace("Levle 3", "Level 3", regex=False)
    .str.replace("Bodyweight", "Body Weight", regex=False)
    .replace(na_tokens, pd.NA))

df["force_technique"] = df["use_of_force_type"].str.replace(r"^Level \d+\s+", "", regex=True)

# clean Use of Force Reason casing & single NaN
df["use_of_force_reason"] = (df["use_of_force_reason"].astype(str).str.strip()
    .replace({"brandishing a weapon": "Brandishing a Weapon"})
    .replace(na_tokens, pd.NA))

# create boolean Subject Arrested column
df["subject_arrested"] = df["subject_arrested"].astype(str).str.strip().str.title().replace(na_tokens, pd.NA)
df["subject_arrested_bool"] = df["subject_arrested"].map({"Yes": True, "No": False})

# standardize unknowns to NaNs
cat_fill_cols = ["race", "sex", "subject_role", "officer_role", "subject_altered_state"]
for col in cat_fill_cols:
    df[col] = df[col].astype("string").str.strip().replace(na_tokens, pd.NA)


# helper column for counting unique subjects per incident (for cleaning the long format)
df["n_subjects_in_incident"] = df.groupby("incident_number")["subject_incident_number"].transform("nunique")

# --- verification check ---
print("All Columns (Original & New):\n", df.columns.tolist())
print("\nForce Level counts:\n", df["use_of_force_level"].value_counts(dropna=False))
print("\nAge summary:\n", df["subject_age"].describe())
print("\nMissing values remaining:\n", df.isna().sum()[df.isna().sum() > 0])

All Columns (Original & New):
 ['incident_number', 'subject_incident_number', 'sex', 'race', 'subject_role', 'subject_age', 'block_address', 'city', 'use_of_force_number', 'officer_incident_number', 'officer_role', 'use_of_force_level', 'use_of_force_type', 'occurred_date', 'subject_arrested', 'use_of_force_reason', 'subject_altered_state', 'non_berkeley_address', 'force_level_num', 'force_technique', 'subject_arrested_bool', 'n_subjects_in_incident']

Force Level counts:
 use_of_force_level
Level 1    5022
Level 2    2120
Level 3     378
NaN           8
Level 4       2
Name: count, dtype: int64

Age summary:
 count    7264.000000
mean       39.413684
std        13.693111
min         2.000000
25%        30.000000
50%        37.000000
75%        47.000000
max       118.000000
Name: subject_age, dtype: float64

Missing values remaining:
 subject_incident_number      31
sex                          48
race                         87
subject_role                 46
subject_age             

In [6]:
df.head()

,incident_number,subject_incident_number,sex,race,subject_role,subject_age,block_address,city,use_of_force_number,officer_incident_number,...,use_of_force_type,occurred_date,subject_arrested,use_of_force_reason,subject_altered_state,non_berkeley_address,force_level_num,force_technique,subject_arrested_bool,n_subjects_in_incident
21,1463,891,Male,Black,Arrested,50.0,0 Block of Seawall,Berkeley,1075,2371,...,Level 1 Control Hold PCT,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Control Hold PCT,True,2
22,1463,890,Female,Asian,Arrested,28.0,0 Block of Seawall,Berkeley,1077,2372,...,Level 1 Control Hold PCT,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Control Hold PCT,True,2
23,1463,890,Female,Asian,Arrested,28.0,0 Block of Seawall,Berkeley,1078,2372,...,Level 1 Lowered TGSP,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Lowered TGSP,True,2
24,1463,890,Female,Asian,Arrested,28.0,0 Block of Seawall,Berkeley,1079,2373,...,Level 1 Body Weight,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Body Weight,True,2
25,1463,890,Female,Asian,Arrested,28.0,0 Block of Seawall,Berkeley,1080,2371,...,Level 1 Body Weight,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Body Weight,True,2


## Resolving Long Format

*Recall from EDA on force data:*

Long format in this dataset refers to a relational cross-join table where one row is a combination of $\text{Incident} \times \text{Subject} \times \text{Force Action}$ (rather than each row representing one person or one police incident) 
* Example Incident #4442, 3 people were involved and officers logged 17 force actions, so BPD paired every person with every action ($3 \times 17 = 51$ rows)
* Even in a single-person incident, if 2 officers each use 2 force techniques, that 1 person can get repeated across 4 rows  

#### Plan to Resolve Long Format

*Recall research question:* How do Use_of_Force_Reason and neighborhood context relate to the severity of force (Use_of_Force_Level), and how often do force encounters actually result in an arrest (Subject_Arrested) across different subject demographics (Race, Sex, Subject_Age)?

Because my research question asks about different aspects (force actions vs. people vs. overall incidents), the long format force data can't inform my research question without double-counting. So the plan is to save 2 clean, deduplicated tables (+ keep the full cleaned long table):

**1. use_of_force_actions.csv** (1 row = 1 unique use_of_force_number)

*How to make it:* drop duplicate use_of_force_number rows

*What it answers:* "How does Use of Force Reason relate to Use of Force Level and Force Technique?"; each force action is counted once

In [7]:
# sanity check: verify each use_of_force_number has consistent action attributes across duplicate rows
action_check_cols = ["incident_number", "officer_incident_number", "use_of_force_level", "use_of_force_type", "use_of_force_reason"]

for col in action_check_cols:
    inconsistent = df.groupby("use_of_force_number")[col].nunique(dropna=False)
    n_bad = (inconsistent > 1).sum()
    if n_bad > 0:
        print(f"Warning: {n_bad} force actions have differing values in '{col}'")

In [14]:
# compute an incident-level arrest summary before dropping individual subject columns
df["any_subject_arrested"] = df.groupby("incident_number")["subject_arrested_bool"].transform("any")

# select action-level columns (excluding individual subject columns to avoid arbitrary pairing)
action_cols = ["use_of_force_number", "incident_number", "occurred_date", "block_address", "city", 
               "officer_incident_number", "officer_role", "use_of_force_level", "force_level_num", 
               "use_of_force_type", "force_technique", "use_of_force_reason", "n_subjects_in_incident", 
               "any_subject_arrested"]

df_actions = (df[action_cols].dropna(subset=["use_of_force_number"])
              .drop_duplicates(subset=["use_of_force_number"])
              .sort_values(["occurred_date", "incident_number", "use_of_force_number"])
              .reset_index(drop=True))

# verify uniqueness and inspect
assert df_actions["use_of_force_number"].is_unique, "Duplicate use_of_force_number found."
print(f"Total unique force actions: {len(df_actions):,}")
print(f"Unique incidents represented: {df_actions['incident_number'].nunique():,}")
print("Columns:", df_actions.columns.tolist())
df_actions.head()

Total unique force actions: 5,851
Unique incidents represented: 1,561
Columns: ['use_of_force_number', 'incident_number', 'occurred_date', 'block_address', 'city', 'officer_incident_number', 'officer_role', 'use_of_force_level', 'force_level_num', 'use_of_force_type', 'force_technique', 'use_of_force_reason', 'n_subjects_in_incident', 'any_subject_arrested']


,use_of_force_number,incident_number,occurred_date,block_address,city,officer_incident_number,officer_role,use_of_force_level,force_level_num,use_of_force_type,force_technique,use_of_force_reason,n_subjects_in_incident,any_subject_arrested
0,773,1304,2021-03-01 00:00:00+00,200 Block of Marina,Berkeley,1991,Primary Officer,Level 1,1,Level 1 Grab,Grab,Combative,3,False
1,774,1304,2021-03-01 00:00:00+00,200 Block of Marina,Berkeley,1989,Cover Officer,Level 1,1,Level 1 Grab,Grab,Combative,3,False
2,775,1304,2021-03-01 00:00:00+00,200 Block of Marina,Berkeley,1990,Cover Officer,Level 1,1,Level 1 Grab,Grab,Combative,3,False
3,776,1305,2021-03-02 00:00:00+00,2500 Block of Haste,Berkeley,1996,Primary Officer,Level 1,1,Level 1 Control Hold PCT,Control Hold PCT,Effect an Arrest,2,False
4,777,1306,2021-03-02 00:00:00+00,2700 Block of Milvia,Berkeley,1998,Supervisor,Level 1,1,Level 1 Grab,Grab,Effect an Arrest,1,False


**2. use_of_force_subjects.csv** (1 row = 1 unique person in an incident, i.e., incident_number + subject_incident_number)

*How to make it:* group by (incident_number, subject_incident_number) and summarize each person's encounter:

* Keep their demographics (race, sex, subject_age, subject_role), location (block_address), timestamp (occurred_date), and outcome (subject_arrested, subject_arrested_bool)

* Summarize the force used in that incident: max_force_level (the highest level 1–4 recorded), n_force_actions_in_incident, and n_subjects_in_incident

*What it answers:* "How often do force encounters end in Subject Arrested across Race, Sex, and Age?"

In [9]:
# check how many filtered rows/incidents are missing a subject_incident_number
missing_subj_rows = df["subject_incident_number"].isna().sum()
missing_subj_incs = df.loc[df["subject_incident_number"].isna(), "incident_number"].nunique()
print(f"Rows without Subject ID: {missing_subj_rows} (across {missing_subj_incs} incidents)")

valid_subj = df.dropna(subset=["subject_incident_number"]).copy()

# verify subject attributes are consistent within each (incident, subject) pair
subj_check_cols = ["race", "sex", "subject_age", "subject_role", "subject_arrested"]
for col in subj_check_cols:
    inconsistent = valid_subj.groupby(["incident_number", "subject_incident_number"])[col].nunique(dropna=False)
    n_bad = (inconsistent > 1).sum()
    if n_bad > 0:
        print(f"Warning: {n_bad} subject-incidents have differing values in '{col}'")

Rows without Subject ID: 31 (across 12 incidents)


In [16]:
# aggregate to 1 row per (incident_number, subject_incident_number)
df_subjects = (
    valid_subj.groupby(["incident_number", "subject_incident_number"], as_index=False)
    .agg(
        occurred_date=("occurred_date", "first"),
        block_address=("block_address", "first"),
        city=("city", "first"),
        sex=("sex", "first"),
        race=("race", "first"),
        subject_age=("subject_age", "first"),
        subject_role=("subject_role", "first"),
        subject_arrested=("subject_arrested", "first"),
        subject_arrested_bool=("subject_arrested_bool", "first"),
        n_subjects_in_incident=("n_subjects_in_incident", "first"),
        max_force_level=("force_level_num", "max"),
        n_force_actions_in_incident=("use_of_force_number", "nunique"),
        primary_force_reason=("use_of_force_reason", "first"),
    )
    .sort_values(["occurred_date", "incident_number", "subject_incident_number"])
    .reset_index(drop=True)
)

print("Columns:", df_subjects.columns.tolist())
df_subjects.head()

Columns: ['incident_number', 'subject_incident_number', 'occurred_date', 'block_address', 'city', 'sex', 'race', 'subject_age', 'subject_role', 'subject_arrested', 'subject_arrested_bool', 'n_subjects_in_incident', 'max_force_level', 'n_force_actions_in_incident', 'primary_force_reason']


,incident_number,subject_incident_number,occurred_date,block_address,city,sex,race,subject_age,subject_role,subject_arrested,subject_arrested_bool,n_subjects_in_incident,max_force_level,n_force_actions_in_incident,primary_force_reason
0,1304,787,2021-03-01 00:00:00+00,200 Block of Marina,Berkeley,Male,Black,46.0,Suspect,No,False,3,1,3,Combative
1,1304,788,2021-03-01 00:00:00+00,200 Block of Marina,Berkeley,Female,Black,33.0,Witness,No,False,3,1,3,Combative
2,1304,789,2021-03-01 00:00:00+00,200 Block of Marina,Berkeley,Female,Black,30.0,Reporting Party,No,False,3,1,3,Combative
3,1305,790,2021-03-02 00:00:00+00,2500 Block of Haste,Berkeley,Male,Black,52.0,Arrested,No,False,2,1,1,Effect an Arrest
4,1305,791,2021-03-02 00:00:00+00,2500 Block of Haste,Berkeley,Male,Black,46.0,Involved Party,No,False,2,1,1,Effect an Arrest


## Export Cleaned Datasets

In [17]:
df_actions.to_csv("../data/clean/use_of_force_actions.csv", index=False)
df_subjects.to_csv("../data/clean/use_of_force_subjects.csv", index=False)
df.to_csv("../data/clean/use_of_force_clean_long.csv", index=False)